# 📚 Week 07 - Day 04: Natural Language Processing (NLP) using Artificial Neural Networks (ANN)

Welcome to today's session! Today we will learn how to teach a Neural Network to understand text. We will build a **Sentiment Analysis** model to predict product ratings based on customer reviews.

### 🎯 Learning Objectives:
1. Understand **Tokenization** (converting text to numbers).
2. Understand **Padding** (making all texts the same length).
3. Learn about **Word Embeddings** (giving meaning to numbers).
4. Build and train an **ANN** for text regression and classification.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

## 1️⃣ Step 1: Loading the Toy Dataset
We will start with a mock dataset of 1,000 product reviews to understand the basics. Our goal is to predict the `Rating` (1 to 5) from the `ReviewText`.


In [ ]:
df = pd.read_csv(r"D:\NAVTTC-AI-Course\datasets\product_reviews_mock_data.csv")
df.head()

In [ ]:
# Let's extract our input features (X) and target labels (y)
X = df["ReviewText"].values
y = df["Rating"].values

print(f"First review: '{X[0]}'")
print(f"Rating: {y[0]}/5")

## 2️⃣ Step 2: Tokenization (Words to Numbers)
Machine Learning models can't read text directly; they only understand numbers. 
**Tokenization** is the process of mapping every unique word to a specific integer index.


In [ ]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(X)
vocab_size = len(tokenizer.word_index) + 1
print(f"Total unique words in vocabulary: {vocab_size}")

In [ ]:
sequences = tokenizer.texts_to_sequences(X)
print("Original Text:", X[0])
print("Tokenized Sequence:", sequences[0])

## 3️⃣ Step 3: Padding (Uniform Sequence Lengths)
Neural networks require standard shapes. We **pad** sentences (add zeroes at the end) or truncate them so they are all the exact same size.


In [ ]:
max_length = 15
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding="post", truncating="post")
print("Padded Sequence Shape:", padded_sequences.shape)
print("Padded Sequence:\n", padded_sequences[0])

## 4️⃣ Step 4: Train / Test Split
We split our data to evaluate how well our model performs on unseen data.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, y, test_size=0.2, random_state=42
)
print("Training data size:", X_train.shape)
print("Testing data size:", X_test.shape)

## 5️⃣ Step 5: Building the ANN Model with Embeddings
An **Embedding Layer** maps each word ID to a dense vector of floating-point values, allowing the network to mathematically learn similarities between words!


In [ ]:
embedding_dim = 16

model = models.Sequential([
    # 1. Embedding Layer: Converts token indices into dense word vectors
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    
    # 2. Flatten: Flattens the 2D embedding into 1D so the Dense layer can process it
    layers.Flatten(),
    
    # 3. Hidden Layers: Standard ANN fully connected layers to learn patterns
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    
    # 4. Output Layer: A single neuron for Regression (predicting continuous 1-5 rating)
    layers.Dense(1)
])

# Compile the model for regression using Mean Squared Error (MSE)
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

## 6️⃣ Step 6: Training and Evaluation
Let's teach the model!


In [ ]:
print("Training the model...")
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

In [ ]:
# Plotting the training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss (MSE)')
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)')
plt.title('Loss Over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('Mean Absolute Error Over Epochs')
plt.legend()

plt.show()

In [ ]:
# Testing the model on a few validation samples
predictions = model.predict(X_test[:5])

print("\n--- Sample Predictions ---")
for i in range(5):
    print(f"Real Rating: {y_test[i]} | Predicted: {predictions[i][0]:.2f}")

## 7️⃣ Bonus Challenge: Try It On Yourself! 
Let's write our own reviews and pass them through our pipeline!


In [ ]:
def predict_review(text):
    seq = tokenizer.texts_to_sequences([text])
    pad_seq = pad_sequences(seq, maxlen=max_length, padding="post", truncating="post")
    pred = model.predict(pad_seq, verbose=0)
    print(f"Review: '{text}' --> Predicted Rating: {pred[0][0]:.2f} / 5.0")

predict_review("This product is absolutely amazing and I love it!")
predict_review("Terrible experience, completely broke on day one.")
predict_review("It's okay, nothing special but gets the job done.")

## 🌍 Extending to Real-World Datasets
Now that you know the pipeline (Tokenization $\rightarrow$ Padding $\rightarrow$ Embedding $\rightarrow$ Dense), let's apply it to a massive dataset! 

We will download a **Twitter Sentiment Analysis** dataset (27,000 tweets) and build an ANN to classify text into `Negative`, `Neutral`, or `Positive`.


In [ ]:
import kagglehub

print("Downloading Kaggle Dataset...")
path = kagglehub.dataset_download("abhi8923shriv/sentiment-analysis-dataset")
print("Path to dataset files:", path)

In [ ]:
# Load and explore the new dataset
train_df = pd.read_csv(r'D:\NAVTTC-AI-Course\datasets\Sentiment Analysis Dataset\train.csv', encoding='latin1')
print(f"Real world dataset loaded! Size: {train_df.shape}")

train_df[['text', 'sentiment']].head()

### 🧹 1. Cleaning and Preprocessing the Real Data
Real-world data is messy. We need to handle missing values (NaNs) and encode text labels (`positive`, `neutral`, `negative`) into numbers (`2`, `1`, `0`).


In [ ]:
# 1. Drop missing rows
train_df.dropna(subset=['text', 'sentiment'], inplace=True)
print(f"Size after dropping NaNs: {train_df.shape}")

# 2. Map textual sentiments to integers (0, 1, 2) since Neural Networks output numbers
sentiment_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
train_df['sentiment_encoded'] = train_df['sentiment'].map(sentiment_mapping)

train_df[['text', 'sentiment', 'sentiment_encoded']].head()

### 📝 2. Tokenization and Padding (Twitter Data)
We need a brand new Tokenizer because Twitter data uses completely different slang, hashtags, and words compared to product reviews.


In [ ]:
# We limit the vocabulary to the top 10,000 most frequent words to avoid crashing memory
twitter_vocab_size = 10000
twitter_max_length = 30 # Tweets are slightly longer than our previous reviews

# 1. Tokenizer
twitter_tokenizer = Tokenizer(num_words=twitter_vocab_size, oov_token="<OOV>")
twitter_tokenizer.fit_on_texts(train_df['text'])

# 2. Convert to Sequences
X_twitter_seq = twitter_tokenizer.texts_to_sequences(train_df['text'])

# 3. Pad Sequences
X_twitter_padded = pad_sequences(X_twitter_seq, maxlen=twitter_max_length, padding="post", truncating="post")
y_twitter = train_df['sentiment_encoded'].values

print("X Tensor shape:", X_twitter_padded.shape)
print("y Tensor shape:", y_twitter.shape)

### ✂️ 3. Train / Test Split


In [ ]:
X_train_tw, X_test_tw, y_train_tw, y_test_tw = train_test_split(
    X_twitter_padded, y_twitter, test_size=0.2, random_state=42
)
print(f"Training set size: {X_train_tw.shape[0]} tweets")
print(f"Test set size: {X_test_tw.shape[0]} tweets")

### 🧠 4. Building a Multi-Class ANN 
Instead of predicting a single number (regression), we are now predicting **3 categories**. 
Therefore, our output layer will have `3 neurons`, and we will use the `softmax` activation to convert those 3 signals into **probabilities** (e.g., 80% confident it's positive, 10% neutral, 10% negative).

*Note: We are using `GlobalAveragePooling1D()` here instead of `Flatten()`. It averages all the word embeddings together, which reduces the number of parameters and speeds up training on larger inputs!*


In [ ]:
twitter_model = models.Sequential([
    layers.Embedding(input_dim=twitter_vocab_size, output_dim=32, input_length=twitter_max_length),
    
    # Averages the embeddings across the time dimension (reduces parameters significantly)
    layers.GlobalAveragePooling1D(),
    
    layers.Dense(64, activation="relu"),
    
    # 3 categories (negative=0, neutral=1, positive=2)
    layers.Dense(3, activation="softmax") 
])

# We use sparse_categorical_crossentropy because our labels are single integers (0, 1, 2) rather than one-hot arrays
twitter_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
twitter_model.summary()

### 🚀 5. Training the Twitter Sentiment Model


In [ ]:
print("Training the Twitter Sentimental ANN...")
history_twitter = twitter_model.fit(
    X_train_tw, y_train_tw,
    epochs=10, # Keep epochs low to save time during lecture
    batch_size=64,
    validation_data=(X_test_tw, y_test_tw),
    verbose=1
)

### 📊 6. Evaluating Multi-Class Performance


In [ ]:
# Plotting the accuracy history
plt.figure(figsize=(10, 4))

plt.plot(history_twitter.history['accuracy'], label='Training Accuracy')
plt.plot(history_twitter.history['val_accuracy'], label='Validation Accuracy')
plt.title('Twitter Model Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

### 💡 7. Testing Fake Tweets! 
Let's test our Multi-Class Sentiment Classifier on a few live inputs.


In [ ]:
def predict_tweet_sentiment(text):
    seq = twitter_tokenizer.texts_to_sequences([text])
    pad_seq = pad_sequences(seq, maxlen=twitter_max_length, padding="post", truncating="post")
    pred = twitter_model.predict(pad_seq, verbose=0)
    
    class_idx = np.argmax(pred[0])  # Find the index with the highest probability
    confidence = np.max(pred[0]) * 100
    
    sentiment_map = {0: "Negative 😡", 1: "Neutral 😐", 2: "Positive 😁"}
    
    print(f"Tweet: '{text}'")
    print(f"--> Sentiment: {sentiment_map[class_idx]} ({confidence:.1f}% confidence)\n")

predict_tweet_sentiment("I had the absolute worst day today, my car broke down and it was raining.")
predict_tweet_sentiment("Just bought some groceries.")
predict_tweet_sentiment("Wow! The new AI course at NAVTTC is fantastic, I'm learning so much!")